# How Many Generations Does Relatedness-Driven Similarity Last? (vem 1uM, 144hrs Pre-Treatment)

**Prototype.** `01_relatedness.ipynb` found that sisters (degree 1: common parent) and cousins (degree 2: common grandparent) are both more morphologically similar than random, for before-drug, after-drug, and naive. This notebook asks the natural follow-up: how far does that similarity reach - does it still hold for second cousins (degree 3: common great-grandparent), third cousins (degree 4), and so on, or does it fade out after a few generations?

`shared/lineage_tree.py`'s `find_relative_pairs(segments, degree)` generalizes `find_sister_pairs`/`find_cousin_pairs` to any `degree` (1 = sisters, 2 = cousins, 3 = second cousins, ...). For each `degree`, this notebook runs the exact same distribution-vs-null Mann-Whitney U test as `01_relatedness.ipynb`, once per (modality, group) - **one plot per modality, 3 lines each** (before-drug, after-drug, naive): x-axis is `degree`, y-axis is `-log10(p-value)` (higher = more significant), with a horizontal line at `p = 0.05` so each line's crossing point is visible directly.

**The real constraint is sample size, not the method.** The number of degree-`d` pairs whose BOTH sides fall inside a group's frame window shrinks fast as `d` grows - especially after-drug, where persisters/dying cells barely divide. A line stops because it ran out of qualifying pairs (printed, and visible as the line simply ending) is a *different* conclusion from a line that keeps going but crosses above `p = 0.05` (relatedness-driven similarity genuinely fades) - the printed `n_pairs` per point is what tells these apart, so read it alongside the plot rather than just the crossing point.

Requires `main.py` to have already been run for BOTH conditions (`vem` and `naive`, for each one's own `selected_metadata.csv` and `{cell,nuc}_qgw_075P_dists.csv` under its own `output/{condition}/cajal_output/`).

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# assumes Jupyter's cwd is this notebook's own directory
# (analyses/04_vem_144hr_lineage_relatedness/notebooks/), so the analysis
# dir - and config.py - is one level up, and the repo root - where
# shared/ lives - is two more levels up from there
NOTEBOOK_DIR = os.getcwd()
ANALYSIS_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
REPO_ROOT    = os.path.abspath(os.path.join(ANALYSIS_DIR, '..', '..'))
sys.path.insert(0, REPO_ROOT)
sys.path.insert(0, ANALYSIS_DIR)
import config

# config.py's OUTPUT_DIR is relative to the analysis directory, not this
# notebook's own cwd - re-anchor it to an absolute path once, here.
# config.cajal_output_dir()/output_dir() both derive from this one
# variable, so re-anchoring it here is enough for every condition's own
# subtree too.
config.OUTPUT_DIR = os.path.join(ANALYSIS_DIR, config.OUTPUT_DIR)

from shared.cajal.embed import load_distance_matrix
from shared.load_data import parse_label
from shared.lineage_tree import build_segments, find_relative_pairs, real_index_pairs, pool_in_range
from shared.pair_similarity import pair_distance_permutation_test

## Method

1. **Reconstruct each condition's own lineage tree** once (same as
   `01_relatedness.ipynb`: `vem` from its FULL metadata in frames
   1-`config.AFTER_DRUG_END_FRAME`, `naive` from its own FULL metadata
   in frames 1-`config.BEFORE_DRUG_END_FRAME` - not just the
   CAJAL-sampled cells).
2. **For each modality** (cell, nucleus) **and condition** (`vem`,
   `naive`), load that condition's own real distance matrix once.
3. **For `degree` = 1, 2, 3, ...**: find every degree-`degree` relative
   pair across each CONDITION's own tree (`find_relative_pairs`) - this
   depends on condition but not modality or group, so it's done once
   per (condition, degree), not once per (modality, group). Then,
   separately for each (modality, group) - before-drug/after-drug (both
   `vem`), naive:
   - Restrict to whichever of each pair's own observations fall in that
     group's own frame range AND survived CAJAL subsampling
     (`real_index_pairs`, exactly as in `01_relatedness.ipynb`) - every
     surviving combination counts as its own pair, not just one per
     relationship.
   - If this (modality, group) has zero surviving pairs at this degree
     (or its own condition's tree hasn't divided this many times yet),
     it has run out of data - stop extending that one line (the others
     may continue).
   - Otherwise, compare the real pairs' own distance distribution
     against a null distribution of individual random-pair distances
     from that group's own pool, via a one-sided Mann-Whitney U test
     (`pair_distance_permutation_test`).
4. **Plot**: one figure PER MODALITY, each with one line per group (3
   total: before-drug, after-drug, naive) of `-log10(p-value)` vs.
   `degree`, with a horizontal line at `p = 0.05`.

In [ ]:
PLOTS_DIR = os.path.join(config.OUTPUT_DIR, 'plots', '02_relatedness_decay')
SVG_DIR = os.path.join(PLOTS_DIR, 'svgs')
os.makedirs(SVG_DIR, exist_ok=True)

MODALITY_NAMES = {'cell': 'cell', 'nuc': 'nucleus'}
GROUP_COLORS = {'before-drug': 'tab:blue', 'after-drug': 'tab:orange', 'naive': 'tab:green'}

# group -> (condition, (start, end) frame, inclusive) - same groups as
# 01_relatedness.ipynb: before-drug/after-drug split vem's ONE lineage
# tree/CAJAL run into two sub-ranges; naive is a wholly separate
# condition (own tree, own CAJAL run)
GROUPS = {
    'before-drug': ('vem', (1, config.BEFORE_DRUG_END_FRAME)),
    'after-drug':  ('vem', (config.AFTER_DRUG_START_FRAME, config.AFTER_DRUG_END_FRAME)),
    'naive':       ('naive', (1, config.CONDITIONS['naive']['max_frame'])),
}
CONDITIONS_USED = sorted({condition for condition, _ in GROUPS.values()})

# a safety cap, not expected to actually bind - each condition's own
# find_relative_pairs (independent of modality/group) is what actually
# stops the sweep in practice, as soon as that condition's tree hasn't
# divided this many times yet; this just guards against an
# unexpectedly deep tree
MAX_DEGREE = 10

# below this many surviving pairs, a point's p-value is too noisy to
# read as a real trend - flagged in the printed summary, not hidden
MIN_RELIABLE_PAIRS = 5

## Build each condition's lineage tree and load each (modality, condition)'s distance matrix (once)

In [ ]:
segments_by_condition = {}
for condition in CONDITIONS_USED:
    max_frame = config.CONDITIONS[condition]['max_frame']
    full_metadata = pd.read_csv(config.metadata_csv_path(condition))

    # tree-building uses the FULL metadata (every tracked cell, has_mask
    # True or False) - has_mask only reflects whether a usable mask crop
    # could be extracted, not whether the cell/lineage tracking itself
    # exists. Filtering to has_mask=True here would drop intermediate
    # rows a later frame's pred_node_id still needs, breaking chains
    # that are otherwise intact.
    full_metadata = full_metadata[full_metadata['frame'] <= max_frame].reset_index(drop=True)

    segments_by_condition[condition] = build_segments(full_metadata)
    print(f"[{condition}] Built {len(segments_by_condition[condition])} segment(s) across every lineage tree in frames 1-{max_frame}")

modality_data = {}
for modality in ['cell', 'nuc']:
    cond_data = {}
    for condition in CONDITIONS_USED:
        dist_csv = os.path.join(config.cajal_output_dir(condition), f'{modality}_qgw_{config.N_POINTS:03d}P_dists.csv')
        labels, dist_matrix = load_distance_matrix(dist_csv)
        label_to_index = {}
        for i, label in enumerate(labels):
            _, frame, pixel_id = parse_label(label)
            label_to_index[(frame, pixel_id)] = i
        cond_data[condition] = {'dist_matrix': dist_matrix, 'label_to_index': label_to_index}
        print(f"[{modality}/{condition}] {len(labels)} CAJAL-embedded cell(s) total")

    # the null's population is fixed per (modality, group) - it does NOT
    # shrink with degree, only the real pairs being tested against it do
    pool_by_group = {
        group_name: pool_in_range(cond_data[condition]['label_to_index'], frame_range)
        for group_name, (condition, frame_range) in GROUPS.items()
    }

    modality_data[modality] = {'cond_data': cond_data, 'pool_by_group': pool_by_group}

## Sweep: degree = 1, 2, 3, ... for each (modality, group)

In [ ]:
results = []
combos = [(modality, group_name) for modality in ['cell', 'nuc'] for group_name in GROUPS]
active = {combo: True for combo in combos}

for degree in range(1, MAX_DEGREE + 1):
    # topology depends only on (condition, degree), not modality - find
    # once per condition per degree, reused by every group that condition feeds
    topology_pairs_by_condition = {
        condition: find_relative_pairs(segments_by_condition[condition], degree)
        for condition in CONDITIONS_USED
    }
    if all(len(pairs) == 0 for pairs in topology_pairs_by_condition.values()):
        print(f"degree {degree}: no lineage in either condition has divided this many times yet - stopping every line")
        break

    for modality, group_name in combos:
        if not active[(modality, group_name)]:
            continue

        condition, frame_range = GROUPS[group_name]
        topology_pairs = topology_pairs_by_condition[condition]
        if not topology_pairs:
            active[(modality, group_name)] = False
            print(f"  [{modality}/{group_name}] degree {degree}: {condition}'s own tree hasn't divided this many times yet - stopping this line")
            continue

        data = modality_data[modality]['cond_data'][condition]
        pool = modality_data[modality]['pool_by_group'][group_name]
        real_idx = real_index_pairs(topology_pairs, frame_range, data['label_to_index'])

        if len(real_idx) == 0:
            active[(modality, group_name)] = False
            print(f"  [{modality}/{group_name}] degree {degree}: 0 surviving pairs - stopping this line")
            continue

        observed_distances, null_distances, p_value = pair_distance_permutation_test(
            data['dist_matrix'], pool, real_idx, n_null_samples=config.N_NULL_SAMPLES, seed=config.TEST_SEED,
        )
        n_pairs = len(real_idx)
        flag = '  <-- fewer than MIN_RELIABLE_PAIRS, read with caution' if n_pairs < MIN_RELIABLE_PAIRS else ''
        print(f"  [{modality}/{group_name}] degree {degree}: {n_pairs} pair(s), "
              f"mean distance = {observed_distances.mean():.4f}, Mann-Whitney U p = {p_value:.4g}{flag}")

        results.append({
            'modality': modality, 'group': group_name, 'degree': degree,
            'n_pairs': n_pairs, 'mean_distance': observed_distances.mean(), 'p_value': p_value,
        })

    if not any(active.values()):
        break

results_df = pd.DataFrame(results)
print(f"\n{len(results_df)} (modality, group, degree) point(s) total")

## Plot: significance vs. relatedness degree - one figure per modality, 3 lines each (before-drug, after-drug, naive)

No error bars (the uncertainty sources explored for this didn't hold up - see the discussion in chat). A hollow marker instead flags any point where the Mann-Whitney U p-value underflowed to exactly 0.0 (an extremely large effect, rather than a hard "resolution floor" the way the old permutation-count method had one) - that point's exact height is a float-precision floor stand-in, not a precise measurement.

In [ ]:
P_FLOOR = np.finfo(float).tiny
results_df['at_resolution_floor'] = results_df['p_value'] == 0
results_df['neg_log10_p'] = -np.log10(results_df['p_value'].clip(lower=P_FLOOR))

# a small, fixed per-group horizontal offset so points that land on the
# exact same (degree, y) - e.g. two groups both underflowing to p == 0
# at the same degree - don't sit exactly on top of each other
DODGE = {'before-drug': -0.08, 'after-drug': 0.0, 'naive': 0.08}

for modality in ['cell', 'nuc']:
    sub = results_df[results_df['modality'] == modality]
    fig, ax = plt.subplots(figsize=(7.5, 5.5))

    for group_name, group in sub.groupby('group'):
        group = group.sort_values('degree')
        degree = group['degree'].to_numpy() + DODGE[group_name]
        ax.plot(degree, group['neg_log10_p'], color=GROUP_COLORS[group_name], marker='o', label=group_name)

        floored = group[group['at_resolution_floor']]
        if not floored.empty:
            ax.scatter(
                floored['degree'].to_numpy() + DODGE[group_name], floored['neg_log10_p'],
                facecolors='none', edgecolors=GROUP_COLORS[group_name], s=140, linewidths=1.5, zorder=5,
            )

    threshold = -np.log10(0.05)
    ax.axhline(threshold, color='grey', linestyle=':', linewidth=1.5)
    if not sub.empty:
        ax.text(sub['degree'].max() + 0.15, threshold, '  p = 0.05', va='center', ha='left', color='grey')
        ax.set_xticks(sorted(sub['degree'].unique()))

    ax.set_xlabel('relatedness degree (1 = sisters, 2 = cousins, 3 = second cousins, ...)')
    ax.set_ylabel('-log10(p-value)\n(higher = more significantly more similar than a random pair)')
    ax.set_title(f"How far does relatedness-driven similarity reach?\n({MODALITY_NAMES[modality]} morphology - hollow marker = p underflowed to 0)")
    ax.legend()
    fig.tight_layout()

    fig.savefig(os.path.join(PLOTS_DIR, f'relatedness_decay_{modality}.pdf'), bbox_inches='tight')
    fig.savefig(os.path.join(SVG_DIR, f'relatedness_decay_{modality}.svg'), bbox_inches='tight')
    display(fig)   # show it in the notebook's own output, not just save it to disk
    plt.close(fig)
    print(f"[{modality}] Saved to {PLOTS_DIR}")

## Where does each line cross p = 0.05?

The first degree, per (modality, group), where `p_value >= 0.05` - i.e. where relatedness-driven similarity stops being distinguishable from chance. A combo that ran out of pairs before ever crossing is reported separately: that's "not enough data to tell", not "similarity never fades".

In [ ]:
for modality, group_name in combos:
    group = results_df[(results_df['modality'] == modality) & (results_df['group'] == group_name)].sort_values('degree')
    label = f'{MODALITY_NAMES[modality]}, {group_name}'
    if group.empty:
        print(f"{label}: no degree had any surviving pairs at all")
        continue

    lost = group[group['p_value'] >= 0.05]
    max_degree_reached = group['degree'].max()
    ran_out = not active[(modality, group_name)]

    if not lost.empty:
        crossing_degree = int(lost['degree'].iloc[0])
        n_pairs = int(lost['n_pairs'].iloc[0])
        print(f"{label}: crosses p = 0.05 at degree {crossing_degree} (n_pairs = {n_pairs})")
    elif ran_out:
        print(f"{label}: still significant through degree {max_degree_reached}, but ran out of pairs before crossing - "
              f"not enough data to say whether it would eventually cross")
    else:
        print(f"{label}: still significant through degree {max_degree_reached} (the sweep's own cap) - never crossed")

## Export plot data

One row per (modality, group, degree): `n_pairs`, `mean_distance`, `p_value`, `at_resolution_floor`, `neg_log10_p` - everything needed to redraw either plot above or re-derive the crossing points, without rerunning the tests.

In [ ]:
plot_data_path = os.path.join(PLOTS_DIR, 'plot_data.csv')
results_df.to_csv(plot_data_path, index=False)
print(f"Saved plot data ({len(results_df)} rows) to {plot_data_path}")